In [1]:
import pandas as pd, numpy as np
from pathlib import Path
import os
# ROOT dinámico — funciona en local, CI y cualquier entorno
ROOT = Path.cwd()
while not (ROOT / "requirements.txt").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)

SILVER = Path("output/economico/02-silver")
GOLD   = Path("output/economico/03-gold")
GOLD.mkdir(parents=True, exist_ok=True)
print("✅ Rutas OK")

✅ Rutas OK


In [2]:
df_compra   = pd.read_parquet(SILVER / "clean_compraventas_provincia_anio.parquet")
df_pob      = pd.read_parquet(SILVER / "clean_poblacion_provincia_anio.parquet")
df_turismo  = pd.read_parquet(SILVER / "clean_turismo_provincia_anio.parquet")
df_hipoteca = pd.read_parquet(SILVER / "clean_hipotecas_ccaa_anio.parquet")
df_ipv      = pd.read_parquet(SILVER / "clean_ipv_vivienda_ccaa_anio.parquet")

# Filtrar solo años razonables (hasta 2024, sin datos futuros espurios)
for name, df in [("compra", df_compra), ("pob", df_pob),
                 ("turismo", df_turismo), ("hipoteca", df_hipoteca), ("ipv", df_ipv)]:
    print(f"{name}: {df.shape} — años: {sorted(df['anyo'].unique())[-5:]}")

compra: (1040, 3) — años: [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]
pob: (3900, 3) — años: [np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021)]
turismo: (1400, 4) — años: [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]
hipoteca: (423, 3) — años: [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
ipv: (361, 3) — años: [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


In [3]:
# Filtrar años limpios
df_compra  = df_compra[df_compra["anyo"] <= 2024]
df_pob     = df_pob[df_pob["anyo"] <= 2021]
df_turismo = df_turismo[df_turismo["anyo"] <= 2024]
df_turismo = df_turismo.dropna(subset=["pernoctaciones"])

# Join provincia×año
df_prov = (
    df_compra
    .merge(df_pob,    on=["provincia", "anyo"], how="outer")
    .merge(df_turismo, on=["provincia", "anyo"], how="outer")
)

# Compraventas per cápita (donde tengamos ambas)
df_prov["compraventas_per_capita"] = (
    df_prov["compraventas_vivienda"] / df_prov["poblacion_provincia"].replace(0, np.nan)
).round(6)

df_prov = df_prov.sort_values(["provincia", "anyo"]).reset_index(drop=True)
print(f"Gold provincia shape: {df_prov.shape}")
print(f"Cols: {list(df_prov.columns)}")
print(df_prov[df_prov["provincia"] == "Madrid"].head(5).to_string())

Gold provincia shape: (4056, 7)
Cols: ['provincia', 'anyo', 'compraventas_vivienda', 'poblacion_provincia', 'viajeros', 'pernoctaciones', 'compraventas_per_capita']
     provincia  anyo  compraventas_vivienda  poblacion_provincia  viajeros  pernoctaciones  compraventas_per_capita
2262    Madrid  1996                    NaN            5022289.0       NaN             NaN                      NaN
2263    Madrid  1996                    NaN            2411548.0       NaN             NaN                      NaN
2264    Madrid  1996                    NaN            2610741.0       NaN             NaN                      NaN
2265    Madrid  1998                    NaN            5091336.0       NaN             NaN                      NaN
2266    Madrid  1998                    NaN            2444919.0       NaN             NaN                      NaN


In [4]:
df_hipoteca = df_hipoteca[df_hipoteca["anyo"] <= 2024]
df_ipv      = df_ipv[df_ipv["anyo"] <= 2024]

df_ccaa = df_hipoteca.merge(df_ipv, on=["ccaa", "anyo"], how="outer")
df_ccaa = df_ccaa.sort_values(["ccaa", "anyo"]).reset_index(drop=True)

print(f"Gold CCAA shape: {df_ccaa.shape}")
print(f"Cols: {list(df_ccaa.columns)}")
print(df_ccaa.head(5).to_string())

Gold CCAA shape: (410, 4)
Cols: ['ccaa', 'anyo', 'hipotecas_vivienda', 'ipv_vivienda']
        ccaa  anyo  hipotecas_vivienda  ipv_vivienda
0  Andalucía  2003            176726.0           NaN
1  Andalucía  2004            197480.0           NaN
2  Andalucía  2005            241470.0           NaN
3  Andalucía  2006            269178.0           NaN
4  Andalucía  2007            254771.0         38.71


In [5]:
df_prov.to_parquet(GOLD / "gold_economico_provincia_anio.parquet", index=False)
df_prov.to_csv(GOLD    / "gold_economico_provincia_anio.csv",     index=False)

df_ccaa.to_parquet(GOLD / "gold_economico_ccaa_anio.parquet", index=False)
df_ccaa.to_csv(GOLD    / "gold_economico_ccaa_anio.csv",     index=False)

print(f"✅ Guardado en {GOLD}")
print(f"   gold_economico_provincia_anio → {df_prov.shape}")
print(f"   gold_economico_ccaa_anio      → {df_ccaa.shape}")

✅ Guardado en output/economico/03-gold
   gold_economico_provincia_anio → (4056, 7)
   gold_economico_ccaa_anio      → (410, 4)
